<h1> Information Retrieval using ChatGPT and the OpenAI Python API library

In [1]:
import openai
from openai import OpenAI
import pandas as pd

In [2]:
# Prepare the connection through the OpenAI Python API library. Credentials will be read from the environment file by default.
client = OpenAI()

In [3]:
FILE_PATH = f"../data/spotify_reviews_filtered.json"

# Fetch relevant data
try:
    df = pd.read_csv('../data/spotify_reviews.csv')
except Exception as e:
    raise Exception(f"Failed to read CSV file: {e}")

# Filter and format relevant data
df = df.drop(columns=['reviewId', 'userName', 'score', 'thumbsUpCount', 'reviewCreatedVersion'])
df['at'] = pd.to_datetime(df['at'])

# Filter data for relevant timeframe
df = df[df['at'] > '2023-11-01']
df = df[df['at'] < '2024-02-01']
 
# Save data as JSON file
try:
    df.to_json(FILE_PATH, orient='records')
except Exception as e:
    raise Exception(f"Failed to write JSON file: {e}")

In [4]:
system_prompt = """
                You are a helpful assistant analyzing Spotify app reviews. 
                Be precise and focus on specific features which are highlighted in the reviews. 
                Provide 5 to 10 words to explain the particular feature for each prompt.
                Limit your response to the data available.
                Do not provide general information.
                Provide your response in a suitable markdown format.
                """

In [5]:
# Create the Assistant
print("Creating assistant...")
try:
    assistant = client.beta.assistants.create(
        name="Spotify Review Analyzer",
        instructions=system_prompt,
        model="gpt-3.5-turbo",
        tools=[{"type": "file_search"}],
        temperature=0.01
    )
except Exception as e:
    raise Exception(f"Failed to create assistant: {e}")

# Create a vector store to store the data
print("Creating vector store...")
try:
    vector_store = client.beta.vector_stores.create(name=f"Spotify Review Vector Store")
except Exception as e:
    raise Exception(f"Failed to create vector store: {e}")

# Prepare files for upload to OpenAI
file_paths = [FILE_PATH]
file_streams = [open(path, "rb") for path in file_paths]

# Use SDK helper to upload the files, add them to the vector store and poll the status of the file batch for completion.
print("Uploading files...")
try:
    file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id, files=file_streams
)
except Exception as e:
    raise Exception(f"Failed to upload files: {e}")
    
# You can print the status and the file counts of the batch to see the result of this operation.
print("Finished uploading files.")
print(file_batch.status)
print(file_batch.file_counts)

# Update the assistant with the vector store
print("Updating assistant with vector store...")
try:
    assistant = client.beta.assistants.update(
        assistant_id=assistant.id,
        tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
    )
except Exception as e:
    raise Exception(f"Failed to update assistant: {e}")

Creating assistant...
Creating vector store...
Uploading files...
Finished uploading files.
completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)
Updating assistant with vector store...


In [6]:
# Create the Thread
try:
    thread = openai.beta.threads.create()
except Exception as e:
    raise Exception(f"Failed to create thread: {e}")

# Add individual review to the thread
def add_user_message(message):
    try:
        openai.beta.threads.messages.create(
            thread_id=thread.id,
            role="user",
            content=message
        )
    except Exception as e:
        raise Exception(f"Failed to add user message: {e}")

# Function to run the assistant on the thread
def run_assistant():
    try:
        run = openai.beta.threads.runs.create(
            thread_id=thread.id,
            assistant_id=assistant.id
        )
        return run
    except Exception as e:
        raise Exception(f"Failed to run assistant: {e}")

# Function to retrieve and process assistant's response
def get_assistant_response(run_id):
    try:
        messages = openai.beta.threads.messages.list(
            thread_id=thread.id
        )
        response_message = messages.data[0]
        return response_message.content[0].text.value
    except Exception as e:
        raise Exception(f"Failed to get assistant response: {e}")

# Function to add user message and run the assistant
def add_user_message_and_run(user_prompt):
    add_user_message(user_prompt)
    run = run_assistant()
    while run.status != "completed":
        run = openai.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    assistant_response = get_assistant_response(run.id)
    return assistant_response

In [7]:
prompt_likes = """
                Please extract the top 10 specific features that users like the most about the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_likes))

Based on the reviews data, the top 10 specific features that users like the most about the Spotify app are:

1. **Ease of use and music discovery**.
2. **Simple navigation and personalized recommendations**.
3. **Music collection and selection**.
4. **Crossfade feature for smooth transitions**.
5. **Great selection of music and offline listening**.
6. **Custom Playlist options and Family plan**.
7. **Lyrics display for learning and intuitive interface**.
8. **Ability to play songs offline and in the background**.
9. **Extensive song library and tailored playlists**.
10. **Unlimited downloads and intelligent shuffle**.

These features were highlighted as particularly appreciated by users in the reviews of the Spotify app【4:0†source】【4:1†source】【4:2†source】.


In [8]:
prompt_dislikes = """
                    Please extract the top 10 specific features that users dislike the most about the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_dislikes))

Based on the reviews data, the top 10 specific features that users dislike the most about the Spotify app are:

1. **Restrictions on basic features without premium**.
2. **Excessive ads and interruptions**.
3. **Inability to choose specific songs without premium**.
4. **Changes in the like button functionality**.
5. **Podcasts taking a long time to load**.
6. **Issues with the app randomly stopping during playback**.
7. **Difficulties with the login process**.
8. **Intrusive ad strategy and premium pressure**.
9. **Issues with song looping and navigation**.
10. **Smart Shuffle feature causing inconvenience**.

These features were highlighted as particularly disliked by users in the reviews of the Spotify app【8:0†source】【8:1†source】【8:2†source】.


In [9]:
prompt_wants = """
                Please extract the top 10 specific features that users want to see in the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_wants))

Based on the reviews data, the specific features that users want to see in the Spotify app include:

1. **Option to sort/group liked songs by genre**.
2. **Improved sharing options for liked songs playlist**.
3. **Ability to download and share specific song lyrics**.
4. **Better shuffle functionality for large playlists**.
5. **Simplified like system for albums**.
6. **Repeat track button in the notification bar**.
7. **Enhanced ability to play multiple favorite songs multiple times**.
8. **Improved recommendation of songs**.
9. **Option to disable smart shuffle permanently**.
10. **Unlimited listening time for audiobooks**.

These desired features were highlighted by users in the reviews of the Spotify app【12:0†source】【12:1†source】【12:3†source】.


In [10]:
prompt_bugs = """
                Please extract the top 10 specific bugs that users have reported in the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_bugs))

Based on the reviews data, the top 10 specific bugs that users have reported in the Spotify app are:

1. **Songs listed multiple times in library, smart shuffle issues**.
2. **App slowness and freezing on the latest Android version**.
3. **Downloads repeatedly removed upon app opening**.
4. **App crashing after commercials, podcast loading issues**.
5. **Sound quality fixes not available in the free version**.
6. **Autoplay issues with repetitive songs**.
7. **Constant need to reinstall for podcast streaming**.
8. **Issues with the (+) button not adding songs to liked playlist**.
9. **Incorrect song playback and limited skips**.
10. **Storage management problems and app instability**.

These bugs were frequently mentioned by users in the reviews of the Spotify app【16:0†source】【16:1†source】【16:2†source】.
